<div style='background: linear-gradient(135deg, #0f1f3d 0%, #1a3a6b 60%, #2255a4 100%); padding: 40px 48px; border-radius: 10px; box-shadow: 0 4px 20px rgba(0,0,0,0.25);'>

  <p style='color: #7fafd6; margin: 0 0 10px 0; font-size: 0.78em; letter-spacing: 4px; text-transform: uppercase; font-weight: 600;'>Module [X] · Covariance Estimation</p>

  <h1 style='color: #ffffff; font-size: 2.4em; font-weight: 800; margin: 0 0 10px 0; line-height: 1.2;'>[Estimator Name]</h1>

  <p style='color: #a8c8e8; font-size: 1.0em; font-weight: 300; margin: 0 0 20px 0;'>Author: [Your Name]</p>

  <div style='width: 48px; height: 3px; background: #4a9eda; border-radius: 2px;'></div>

</div>


---

## §1 Motivation

*What problem does this estimator solve? Build the intuition before any equations.*

[Write ~200 words. Explain what is wrong with the sample covariance that this estimator addresses. What structural assumption does it make? When would you expect it to work well, and when would it struggle?]


---

## §2 Derivation

*Derive the estimator from first principles. Every step should follow from the one before.*

### Setup

[State the problem. What are we estimating? What loss function or criterion are we optimizing?]

$$\mathcal{L}(\boldsymbol{\Sigma}) = \text{[your objective]}$$

### Key result

[Work through the algebra. End with a closed-form expression or a clear algorithm.]

$$\boxed{\hat{\boldsymbol{\Sigma}} = \text{[your estimator]}}$$

### Interpretation

[In 2–3 sentences: what does the formula tell us geometrically or statistically?]


---

## §3 Implementation


In [ ]:
# ---------------------------------------------------------
# Setup: load shared universe data
# ---------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
import os, warnings
warnings.filterwarnings('ignore')

DATA_DIR = 'data'

if os.path.exists(f'{DATA_DIR}/prices.csv') and os.path.exists(f'{DATA_DIR}/returns.csv'):
    prices  = pd.read_csv(f'{DATA_DIR}/prices.csv',  index_col=0, parse_dates=True)
    returns = pd.read_csv(f'{DATA_DIR}/returns.csv', index_col=0, parse_dates=True)
    print("Loaded data from CSV files")
else:
    print("CSV files not found — run 00_preface_final.ipynb first to generate them.")
    raise FileNotFoundError("Run the preface notebook to download and save the data.")

print(f"Universe: {returns.shape[1]} assets,  {returns.shape[0]} trading days")
print(f"Period  : {returns.index[0].date()} → {returns.index[-1].date()}")


In [ ]:
# ---------------------------------------------------------
# §3 — YOUR ESTIMATOR
# ---------------------------------------------------------

def estimate_covariance(returns_df):
    """
    Estimate the covariance matrix using [YOUR METHOD].

    Parameters
    ----------
    returns_df : pd.DataFrame  (T × N)
        Daily asset returns

    Returns
    -------
    cov : np.ndarray  (N × N)
        Estimated covariance matrix
    """
    # ── TODO: replace with your estimator ──────────────────
    # Placeholder: standard sample covariance
    return returns_df.cov().values


# ── Sanity checks ─────────────────────────────────────────
np.random.seed(0)
R_toy = pd.DataFrame(np.random.randn(200, 10))
C_toy = estimate_covariance(R_toy)

assert C_toy.shape == (10, 10),                     "Shape mismatch"
assert np.allclose(C_toy, C_toy.T, atol=1e-10),    "Not symmetric"
assert np.all(np.linalg.eigvalsh(C_toy) > -1e-9),  "Not positive semi-definite"

print("✓ Estimator passes shape / symmetry / PSD checks on toy data")


---

## §4 Backtest

Rolling monthly backtest comparing three strategies:

1. **Equal-weight** — $w_i = 1/N$, no estimation required  
2. **Sample covariance MVO** — standard baseline  
3. **Your estimator MVO** — the method you implemented above


In [ ]:
# ---------------------------------------------------------
# Portfolio optimization utilities
# ---------------------------------------------------------

def min_variance_portfolio(cov, allow_short=False):
    """
    Compute minimum-variance portfolio weights.

    Parameters
    ----------
    cov : np.ndarray  (N × N)
    allow_short : bool  — if False, enforce w ≥ 0

    Returns
    -------
    w : np.ndarray  (N,)
    """
    n  = cov.shape[0]
    w0 = np.ones(n) / n
    constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1}]
    bounds      = None if allow_short else [(0, 1)] * n

    res = minimize(lambda w: w @ cov @ w, w0,
                   method='SLSQP', bounds=bounds,
                   constraints=constraints,
                   options={'ftol': 1e-12, 'maxiter': 1000})
    return res.x if res.success else w0


def sample_covariance(returns_df):
    return returns_df.cov().values


In [ ]:
# ---------------------------------------------------------
# Rolling backtest engine
# ---------------------------------------------------------

LOOKBACK       = 252   # estimation window (trading days)
REBALANCE_FREQ = 21    # rebalance interval (~monthly)
MIN_HISTORY    = 252   # warm-up period

dates    = returns.index[MIN_HISTORY:]
n_assets = returns.shape[1]

STRATEGIES = ['Equal Weight', 'Sample Cov MVO', 'Your Estimator MVO']
COLORS     = {'Equal Weight': '#888', 'Sample Cov MVO': '#2255a4', 'Your Estimator MVO': '#c53030'}

portfolio_values  = {s: [1.0]               for s in STRATEGIES}
weights_history   = {s: []                  for s in STRATEGIES}
current_weights   = {s: np.ones(n_assets)/n_assets for s in STRATEGIES}

for i, date in enumerate(dates):
    idx = returns.index.get_loc(date)

    if i % REBALANCE_FREQ == 0:
        hist = returns.iloc[idx - LOOKBACK : idx]

        current_weights['Equal Weight']       = np.ones(n_assets) / n_assets
        current_weights['Sample Cov MVO']     = min_variance_portfolio(sample_covariance(hist))
        current_weights['Your Estimator MVO'] = min_variance_portfolio(estimate_covariance(hist))

    for s in STRATEGIES:
        weights_history[s].append(current_weights[s].copy())

    daily_ret = returns.iloc[idx].values
    for s in STRATEGIES:
        portfolio_values[s].append(portfolio_values[s][-1] * (1 + current_weights[s] @ daily_ret))

portfolio_df = pd.DataFrame(
    portfolio_values,
    index=[dates[0] - pd.Timedelta(days=1)] + list(dates)
)
print(f"Backtest complete — {returns.shape[0]-MIN_HISTORY} days evaluated")


---

## §5 Results


In [ ]:
# ---------------------------------------------------------
# Plot: Cumulative performance
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(13, 5))

for s in STRATEGIES:
    lw = 2.2 if 'Your' in s else 1.6
    ax.plot(portfolio_df.index, portfolio_df[s],
            label=s, color=COLORS[s], linewidth=lw)

ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Portfolio value  ($1 initial)', fontsize=11)
ax.set_title('Cumulative Performance: Your Estimator vs Baselines',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.25)
sns.despine(ax=ax)
plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------------------------------
# Performance metrics table
# ---------------------------------------------------------

def compute_metrics(values_series, rf=0.0):
    r = values_series.pct_change().dropna()
    ann_ret = r.mean() * 252
    ann_vol = r.std()  * np.sqrt(252)
    sharpe  = (ann_ret - rf) / ann_vol if ann_vol > 0 else 0
    cummax  = values_series.cummax()
    max_dd  = ((values_series - cummax) / cummax).min()
    return {
        'Ann. Return (%)':    round(ann_ret * 100, 2),
        'Ann. Volatility (%)': round(ann_vol * 100, 2),
        'Sharpe Ratio':       round(sharpe, 3),
        'Max Drawdown (%)':   round(max_dd * 100, 2),
        'Final Value ($)':    round(values_series.iloc[-1], 3),
    }

metrics = pd.DataFrame({s: compute_metrics(portfolio_df[s]) for s in STRATEGIES}).T
print("Performance Summary:\n")
display(metrics)


In [ ]:
# ---------------------------------------------------------
# Plot: Portfolio concentration (HHI) over time
# ---------------------------------------------------------

def hhi(w):
    return float(np.sum(w ** 2))

hhi_df = pd.DataFrame(
    {s: [hhi(w) for w in weights_history[s]] for s in STRATEGIES},
    index=dates
)

fig, ax = plt.subplots(figsize=(13, 4))
for s in STRATEGIES:
    ax.plot(hhi_df.index, hhi_df[s], label=s, color=COLORS[s], alpha=0.85)

ax.axhline(1/n_assets, color='black', linestyle='--', linewidth=1, alpha=0.5,
           label=f'Equal-weight HHI = {1/n_assets:.4f}')
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Herfindahl Index (HHI)', fontsize=11)
ax.set_title('Portfolio Concentration Over Time', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)
sns.despine(ax=ax)
plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------------------------------
# Plot: 63-day rolling volatility
# ---------------------------------------------------------

port_ret   = portfolio_df.pct_change().dropna()
rolling_vol = port_ret.rolling(63).std() * np.sqrt(252) * 100

fig, ax = plt.subplots(figsize=(13, 4))
for s in STRATEGIES:
    ax.plot(rolling_vol.index, rolling_vol[s], label=s, color=COLORS[s], alpha=0.85)

ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Rolling volatility (%, annualised)', fontsize=11)
ax.set_title('63-Day Rolling Volatility', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)
sns.despine(ax=ax)
plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------------------------------
# Performance by market regime
# ---------------------------------------------------------

eq_vol       = port_ret['Equal Weight'].rolling(63).std() * np.sqrt(252)
vol_threshold = eq_vol.quantile(0.7)
masks = {'High-vol periods': eq_vol > vol_threshold,
         'Low-vol periods' : eq_vol <= vol_threshold}

for label, mask in masks.items():
    sub = port_ret[mask].dropna()
    regime = pd.DataFrame({
        s: {
            'Ann. Return (%)':    round(sub[s].mean() * 252 * 100, 2),
            'Ann. Volatility (%)': round(sub[s].std() * np.sqrt(252) * 100, 2),
        }
        for s in STRATEGIES
    }).T
    print(f"\n{label}  ({mask.sum()} days):")
    display(regime)


In [ ]:
# ---------------------------------------------------------
# Plot: Final portfolio weights
# ---------------------------------------------------------

tickers = returns.columns.tolist()
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, s in zip(axes, STRATEGIES):
    w = current_weights[s]
    ax.bar(range(len(w)), w, color=COLORS[s], alpha=0.75)
    ax.set_xticks(range(len(w)))
    ax.set_xticklabels(tickers, rotation=90, fontsize=6)
    ax.set_ylabel('Weight', fontsize=10)
    ax.set_title(f'{s}\nHHI = {hhi(w):.4f}', fontsize=10, fontweight='bold')
    ax.set_ylim(0, max(0.3, w.max() * 1.15))
    sns.despine(ax=ax)

plt.suptitle('Final Portfolio Weights (last rebalance)', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


---

## §6 Interpretation

*Did the theory hold? When should someone use this estimator?*

[Write ~300 words. Structure your answer around these questions:]

1. **Did your estimator beat the sample covariance baseline?** If yes, by how much and why? If not, what went wrong?

2. **When did your estimator perform best vs worst?** Compare high-vol and low-vol periods. Does the pattern match the theory you derived in §2?

3. **What happened to portfolio concentration (HHI)?** Did your estimator produce more or less diversified portfolios than the baseline? Is that expected?

4. **What are the limitations?** In what market conditions or data regimes would you *not* recommend this estimator?

5. **Key takeaway:** in one sentence, what did this module teach you about risk estimation?


---

<p style='text-align: center; color: #999; font-style: italic; font-size: 0.9em;'>
Module [X] complete &nbsp;·&nbsp; See integration notebook for side-by-side comparison with all seven estimators.
</p>
